# NSL-KDD focal-loss MLP benchmark


### Connexion && Synchronisation Drive - Github

In [ ]:
import os
import json
import urllib.request
from pathlib import Path

BENCHMARK_NAME = 'nsl-kdd'
os.environ['BENCHMARK_NAME'] = BENCHMARK_NAME
SETUP_URL = 'https://raw.githubusercontent.com/Appolinairee/-research-scratch-deep-learning-vision/main/notebooks/colab/setup_colab.ipynb'
SETUP_PATH = Path('/content/setup_colab.ipynb')
urllib.request.urlretrieve(SETUP_URL, SETUP_PATH)
setup_notebook = json.loads(SETUP_PATH.read_text())
for setup_cell in setup_notebook['cells']:
    if setup_cell['cell_type'] == 'code':
        exec(compile(''.join(setup_cell['source']), str(SETUP_PATH), 'exec'), globals())


In [ ]:
import subprocess
import time

import matplotlib.pyplot as plt
import numpy as np

from src.data import load_nsl_kdd
from src.data import NSLKDD_CLASSES
from src.evaluation import classification_report
from src.experiments import NSLKDDModelConfig, build_nsl_kdd_cross_entropy, build_nsl_kdd_model
from src.optim import ReduceLROnPlateau, SGD
from src.regularization import L2Regularization
from src.training.sequential import train_multiclass_classifier
from src.core.backend import get_backend

np.set_printoptions(precision=4, suppress=True)

In [ ]:
subprocess.run(
    ['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', 'main'],
    check=True,
)

local_commit = subprocess.check_output(
    ['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'],
    text=True,
).strip()
remote_commit = subprocess.check_output(
    ['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'origin/main'],
    text=True,
).strip()

print('Drive :', local_commit)
print('GitHub:', remote_commit)
print('Aligné :', local_commit == remote_commit)


## NSL-KDD - Network intrusion benchmark

Charger NSL-KDD et encoder ses 41 variables réseau pour le MLP.

**Pourquoi** : reproduire le benchmark DNN de l’article avec notre implémentation dense from scratch.

In [ ]:
import urllib.request

NSL_KDD_URLS = {
    'train': 'https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain%2B.txt',
    'test': 'https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest%2B.txt',
}
for split, url in NSL_KDD_URLS.items():
    destination = DATA_ROOT / f'KDD{split.title()}%2B.txt'
    if not destination.exists():
        urllib.request.urlretrieve(url, destination)
print('NSL-KDD files ready:', DATA_ROOT)

In [ ]:
data = load_nsl_kdd(
    train_path=DATA_ROOT / 'KDDTrain%2B.txt',
    test_path=DATA_ROOT / 'KDDTest%2B.txt',
    validation_fraction=0.2,
    seed=42,
)
print({
    'train': data.X_train.shape,
    'validation': data.X_validation.shape,
    'test': data.X_test.shape,
    'classes': NSLKDD_CLASSES,
})

In [ ]:
counts = np.bincount(data.y_train_idx, minlength=len(NSLKDD_CLASSES))
plt.figure(figsize=(8, 4))
plt.bar(NSLKDD_CLASSES, counts)
plt.title('NSL-KDD - distribution des classes')
plt.ylabel("Nombre d'exemples")
plt.show()

## Configuration de l’expérience

Les opérations de chargement, encodage, normalisation et séparation sont encapsulées dans `src.data.nsl_kdd`.

In [ ]:
config = NSLKDDModelConfig()
BACKEND = 'gpu'
backend = get_backend(BACKEND)

model = build_nsl_kdd_model(
    config=config,
    input_dim=data.X_train.shape[1],
    backend=backend,
)
loss = build_nsl_kdd_cross_entropy(backend=backend)
optimizer = SGD(lr=config.learning_rate, momentum=config.momentum)
regularizer = L2Regularization(strength=config.l2_strength)

print({
    'backend': 'CuPy' if backend.is_gpu else 'NumPy',
    'architecture': [data.X_train.shape[1], *config.hidden_units, len(NSLKDD_CLASSES)],
    'loss': 'multiclass cross-entropy',
    'epochs': config.epochs,
    'batch_size': config.batch_size,
})

## Modèle et entraînement

Le notebook orchestre uniquement l’expérience ; la construction du MLP et de la loss est testée dans `src.experiments.nsl_kdd`.

In [ ]:
start_time = time.perf_counter()
scheduler = ReduceLROnPlateau(factor=0.5, patience=10, min_lr=1e-6)
history = train_multiclass_classifier(
    model=model,
    X=data.X_train,
    y=data.y_train,
    loss=loss,
    optimizer=optimizer,
    epochs=config.epochs,
    batch_size=config.batch_size,
    shuffle=True,
    seed=42,
    validation_data=(data.X_validation, data.y_validation),
    regularizer=regularizer,
    scheduler=scheduler,
)
backend.synchronize()
elapsed_seconds = time.perf_counter() - start_time
print({'elapsed_seconds': round(elapsed_seconds, 2), 'best_epoch': history.best_epoch})

## Courbes d’apprentissage

In [ ]:
epochs = [row.epoch for row in history]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, [row.loss for row in history], label='train')
axes[0].plot(epochs, [row.validation_loss for row in history], label='validation')
axes[0].set_title('Loss')
axes[0].legend()
axes[1].plot(epochs, [row.accuracy for row in history], label='train')
axes[1].plot(epochs, [row.validation_accuracy for row in history], label='validation')
axes[1].set_title('Accuracy')
axes[1].legend()
plt.show()

## Évaluation finale

In [ ]:
report = classification_report(
    model=model,
    X=data.X_test,
    y=data.y_test,
    class_count=len(NSLKDD_CLASSES),
    batch_size=config.batch_size,
)
print({
    'accuracy': round(report.accuracy, 4),
    'macro_precision': round(report.macro_precision, 4),
    'macro_recall': round(report.macro_recall, 4),
    'macro_f1': round(report.macro_f1, 4),
})
print('Le test n’a pas été utilisé pour la sélection.')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
image = ax.imshow(report.confusion_matrix, cmap='Blues')
ax.set_title('NSL-KDD - matrice de confusion')
ax.set_xlabel('Classe prédite')
ax.set_ylabel('Classe réelle')
ax.set_xticks(range(len(NSLKDD_CLASSES)), NSLKDD_CLASSES, rotation=45)
ax.set_yticks(range(len(NSLKDD_CLASSES)), NSLKDD_CLASSES)
fig.colorbar(image, ax=ax)
plt.show()

### Lecture des résultats

La métrique principale est le macro-F1 : elle donne le même poids aux classes rares et majoritaires.

In [ ]:
print('Confusion matrix:')
print(report.confusion_matrix)